# Notebook T4.6 — Evaluación Automática con LLM-as-Judge
## Patrón: "Cómo medir calidad cuando no hay ground truth fácil"

**Referencia en la guía metodológica:** PASO 6 → Evaluación con LLM-Judge  
**Nivel:** Intermedio-Avanzado

---

### El problema de evaluación en GenAI

En ML clásico evaluar es sencillo: tienes labels → calculas accuracy/F1/AUC.  

En GenAI el problema es diferente:
- La salida es **texto libre** — no hay un único "correcto"
- ¿Cómo sabes si "el aprendizaje supervisado es cuando..." es mejor que "el aprendizaje supervisado consiste en..."?
- Contratar anotadores humanos es caro ($$ por pregunta) y lento

**LLM-as-Judge** es la solución estándar de la industria:  
usas un LLM separado para evaluar la calidad de las respuestas de otro LLM.  
Es más rápido que humanos, más barato, y más consistente si el prompt del juez está bien diseñado.

---

### ¿Qué más aprenderás?

- Cómo hacer un **A/B test de prompts** (prompt básico vs prompt optimizado)
- Por qué **ROUGE** (métrica clásica de NLP) es insuficiente para GenAI
- Cómo construir un **sistema de evaluación continua** reutilizable

---

### Conexión con la guía metodológica

- **Paso 6:** LLM-as-Judge para evaluar cuando no hay ground truth binario  
- **Paso 7:** A/B test de prompts — base del ciclo de mejora continua  
- **Paso 8:** Función `evaluar_pipeline_llm()` reutilizable para cualquier chain

## PASO 2 — Instalación y Configuración

### Dos instancias del mismo LLM con distinta configuración

Usaremos **el mismo modelo** en dos roles:
- `llm_generador` (temperatura 0.7): genera respuestas con algo de creatividad y variedad
- `llm_juez` (temperatura 0.0): evalúa de forma estricta y determinista

Esta separación es importante: el mismo modelo puede hacer ambas cosas,  
pero la temperatura correcta para cada rol es muy diferente.

In [ ]:
!pip install langchain langchain-google-genai pandas numpy python-dotenv rouge-score -q

In [ ]:
import os
import json
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# El generador tiene temperatura alta para producir respuestas variadas
llm_generador = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GOOGLE_API_KEY,
    temperature=0.7  # creatividad para generar respuestas diversas
)

# El juez tiene temperatura CERO para evaluar de forma consistente y reproducible
llm_juez = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GOOGLE_API_KEY,
    temperature=0.0  # determinista para evaluación justa
)

print("✅ Dos instancias configuradas:")
print("  llm_generador: temperatura=0.7  (genera respuestas creativas)")
print("  llm_juez:      temperatura=0.0  (evalúa de forma determinista)")

## PASO 4 — Benchmark de Evaluación

### ¿Qué es un benchmark?

Un **benchmark** es un conjunto de preguntas con respuestas de referencia (ground truth)  
que usas para evaluar distintos modelos, prompts o sistemas de forma comparable.

En este caso, el benchmark contiene:
- La **pregunta** que se le hace al LLM
- El **contexto** del estudiante que la hace (afecta el nivel de la respuesta esperada)
- La **respuesta de referencia** escrita manualmente — el "gold standard"
- Los **criterios** específicos que el juez debe evaluar

### Diferencia entre respuesta de referencia y respuesta perfecta

La respuesta de referencia NO es la única respuesta correcta — es el punto de comparación.  
El LLM-as-Judge debe evaluar si la respuesta candidata es equivalente o mejor, no si es idéntica.

In [ ]:
benchmark = [
    {
        "id": 1,
        "pregunta": "¿Qué es el aprendizaje supervisado en Machine Learning?",
        "contexto": "Estudiante universitario de primer año",
        "respuesta_referencia": (
            "El aprendizaje supervisado es un tipo de ML donde el modelo aprende "
            "a partir de datos etiquetados (inputs + outputs correctos). "
            "El modelo aprende a predecir el output para nuevos inputs. "
            "Ejemplos: clasificación de spam, predicción de precios de vivienda."
        ),
        "criterios": ["precision_tecnica", "claridad", "ejemplos_concretos"]
    },
    {
        "id": 2,
        "pregunta": "Explica la diferencia entre overfitting y underfitting",
        "contexto": "Persona que está aprendiendo ML",
        "respuesta_referencia": (
            "Overfitting: el modelo memoriza los datos de entrenamiento (incluyendo ruido) "
            "y falla en datos nuevos — aprende demasiado bien lo específico. "
            "Underfitting: el modelo es demasiado simple y no capta los patrones — aprende poco. "
            "El objetivo es encontrar el equilibrio que generalice bien a datos no vistos."
        ),
        "criterios": ["precision_tecnica", "claridad", "analogia_o_ejemplo"]
    },
    {
        "id": 3,
        "pregunta": "¿Cuándo usar un árbol de decisión vs una red neuronal?",
        "contexto": "Selección práctica de modelo en un proyecto",
        "respuesta_referencia": (
            "Árbol de decisión: datos tabulares pequeños/medianos, necesitas interpretabilidad, "
            "recursos limitados, pocas features. "
            "Red neuronal: datos masivos, imágenes/texto/audio, patrones muy complejos, "
            "tienes GPU disponible."
        ),
        "criterios": ["precision_tecnica", "practicidad", "casos_de_uso"]
    },
    {
        "id": 4,
        "pregunta": "¿Qué es un pipeline en ML y para qué sirve?",
        "contexto": "Explicación técnica con ejemplos",
        "respuesta_referencia": (
            "Un pipeline ML encadena pasos de preprocesamiento y modelado de forma reproducible. "
            "Evita data leakage y facilita el despliegue en producción. "
            "En sklearn: Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression())]). "
            "El fit/transform se aplica de forma consistente a train y test."
        ),
        "criterios": ["precision_tecnica", "ejemplo_practico", "ventajas"]
    },
]

print(f"Benchmark cargado: {len(benchmark)} preguntas de evaluación")
print(f"\nPregunta 1: {benchmark[0]['pregunta']}")

## PASO 5 — A/B Test de Prompts: Básico vs Optimizado

### ¿Qué es un A/B test de prompts?

Un **A/B test de prompts** compara dos versiones de un prompt para la misma tarea:
- **Variante A (básica)**: el prompt más simple posible
- **Variante B (optimizada)**: con contexto, rol, y restricciones de formato

La hipótesis del módulo es que añadir contexto y estructura al prompt mejora la calidad.  
El LLM-as-Judge lo medirá objetivamente.

### Diferencia entre las variantes

```
Prompt A (básico):
  "Responde esta pregunta: {pregunta}"
  → Sin rol, sin contexto, sin restricciones de longitud

Prompt B (optimizado):
  "Eres un profesor experto. Contexto: {contexto}
   Responde de forma clara, con ejemplo, en máximo 4 oraciones."
  → Con rol, con contexto del estudiante, con restricciones
```

In [ ]:
# Variante A: prompt básico — mínimo contexto, sin restricciones
prompt_basico = ChatPromptTemplate.from_template(
    "Responde esta pregunta de Machine Learning: {pregunta}"
)

# Variante B: prompt optimizado — con rol, contexto y restricciones
prompt_optimizado = ChatPromptTemplate.from_template("""
Eres un profesor experto en Machine Learning.
Contexto del estudiante que pregunta: {contexto}

Pregunta: {pregunta}

Responde de forma clara, con al menos un ejemplo concreto,
y en no más de 4 oraciones. Usa lenguaje accesible.
""")

chain_a = prompt_basico | llm_generador | StrOutputParser()
chain_b = prompt_optimizado | llm_generador | StrOutputParser()

# Generar respuestas para todas las preguntas del benchmark
print("Generando respuestas con ambas variantes de prompt...\n")
respuestas = []

for item in benchmark:
    print(f"  Pregunta {item['id']}: {item['pregunta'][:50]}...", end=" ")

    resp_a = chain_a.invoke({"pregunta": item["pregunta"]})
    resp_b = chain_b.invoke({"pregunta": item["pregunta"], "contexto": item["contexto"]})

    respuestas.append({
        "id": item["id"],
        "pregunta": item["pregunta"],
        "respuesta_referencia": item["respuesta_referencia"],
        "criterios": item["criterios"],
        "respuesta_a": resp_a,
        "respuesta_b": resp_b,
    })
    print("✅")

print(f"\n✅ {len(respuestas) * 2} respuestas generadas ({len(respuestas)} por cada variante)")
print(f"\nEjemplo — Pregunta 1:")
print(f"  RESP A (básico):     {respuestas[0]['respuesta_a'][:100]}...")
print(f"  RESP B (optimizado): {respuestas[0]['respuesta_b'][:100]}...")

## PASO 6 — LLM-as-Judge: El Evaluador Automático

### Diseño del prompt del juez

El prompt del juez es crítico. Debe:
1. Dar un **rol de evaluador experto** — no un generador
2. Proporcionar tanto la **referencia** como la **respuesta a evaluar**
3. Pedir una evaluación **multidimensional** (no solo un número)
4. Especificar el **formato JSON** para parsear el resultado
5. Incluir los **criterios específicos** de cada pregunta

### Sesgo del LLM-as-Judge

El LLM-as-Judge tiene sesgos conocidos:
- **Sesgo de posición**: tiende a preferir la primera respuesta que lee
- **Sesgo de longitud**: tiende a preferir respuestas más largas
- **Sesgo self-preferencia**: si el juez es el mismo modelo que generó la respuesta

En producción, mitigas estos sesgos usando modelos distintos para generar y juzgar,
y evaluando las respuestas en orden aleatorio.

In [ ]:
prompt_juez = ChatPromptTemplate.from_template("""
Eres un evaluador experto en pedagogía y Machine Learning.
Evalúa la calidad de esta respuesta según los criterios dados.

**Pregunta:** {pregunta}
**Respuesta de referencia (gold standard):** {referencia}
**Respuesta a evaluar:** {respuesta}
**Criterios de evaluación:** {criterios}

Evalúa y responde ÚNICAMENTE con este JSON (sin markdown):
{{
  "puntuacion_global": número entre 1 y 10,
  "precision_tecnica": número entre 1 y 10,
  "claridad": número entre 1 y 10,
  "completitud": número entre 1 y 10,
  "fortalezas": "2 fortalezas separadas por coma",
  "debilidades": "2 debilidades o N/A separadas por coma",
  "justificacion": "máximo 2 oraciones explicando la puntuación global"
}}
""")

chain_juez = prompt_juez | llm_juez | StrOutputParser()


def evaluar_respuesta(pregunta: str, referencia: str, respuesta: str, criterios: list) -> dict:
    """Evalúa una respuesta usando LLM-as-Judge. Retorna dict con puntuaciones."""
    raw = chain_juez.invoke({
        "pregunta": pregunta,
        "referencia": referencia,
        "respuesta": respuesta,
        "criterios": ", ".join(criterios)
    })
    raw = raw.strip().replace("```json", "").replace("```", "").strip()
    return json.loads(raw)


# Evaluar todas las respuestas
print("Evaluando respuestas con LLM-as-Judge...\n")
evaluaciones = []

for item in respuestas:
    print(f"  Pregunta {item['id']}:", end=" ")

    eval_a = evaluar_respuesta(item["pregunta"], item["respuesta_referencia"],
                               item["respuesta_a"], item["criterios"])
    eval_b = evaluar_respuesta(item["pregunta"], item["respuesta_referencia"],
                               item["respuesta_b"], item["criterios"])

    evaluaciones.append({
        "id": item["id"],
        "pregunta": item["pregunta"],
        "variante_a": eval_a,
        "variante_b": eval_b,
        "respuesta_a": item["respuesta_a"],
        "respuesta_b": item["respuesta_b"],
    })
    print(f"A={eval_a['puntuacion_global']}/10  B={eval_b['puntuacion_global']}/10 ✅")

print("\n✅ Evaluaciones completadas")

## PASO 6 (continuación) — ROUGE vs LLM-Judge: ¿Por qué ROUGE no es suficiente?

### ¿Qué es ROUGE?

**ROUGE** (Recall-Oriented Understudy for Gisting Evaluation) mide la **similitud léxica** 
entre dos textos: cuántas palabras de la referencia aparecen en la respuesta candidata.

Fue diseñado para evaluar **resúmenes automáticos** en los años 2000, antes de los LLMs.

**El problema**: ROUGE mide si usas las mismas palabras, no si la respuesta es buena.  
- "El ML supervisado usa datos con etiquetas" → ROUGE bajo vs referencia larga
- "El ML supervisado es donde los datos tienen etiquetas (inputs con outputs conocidos)" → ROUGE alto

Pero ambas pueden ser igualmente correctas o incorrectas — ROUGE no lo puede saber.

In [ ]:
def rouge1_simple(hipotesis: str, referencia: str) -> float:
    """
    Implementación simplificada de ROUGE-1: overlap de unigramas entre hipótesis y referencia.
    Un ROUGE-1 de 0.5 significa que el 50% de las palabras de la referencia aparecen en la hipótesis.
    
    Nota: ROUGE real usa precision, recall y F1. Esta es una aproximación educativa (solo recall).
    """
    tokens_hip = set(hipotesis.lower().split())
    tokens_ref = set(referencia.lower().split())
    overlap = len(tokens_hip & tokens_ref)  # palabras en común
    if len(tokens_ref) == 0:
        return 0.0
    return overlap / len(tokens_ref)  # recall: ¿cuántas palabras de la referencia están en hipótesis?


# Comparar métricas: ROUGE vs LLM-Judge
print("=" * 70)
print(f"{'ID':<4} {'ROUGE-A':>8} {'ROUGE-B':>8} {'LLM-A':>7} {'LLM-B':>7}  {'Ganador LLM vs ROUGE'}")
print("-" * 70)

discrepancias = 0
for i, (eval_item, resp_item) in enumerate(zip(evaluaciones, respuestas)):
    rouge_a = rouge1_simple(resp_item["respuesta_a"], resp_item["respuesta_referencia"])
    rouge_b = rouge1_simple(resp_item["respuesta_b"], resp_item["respuesta_referencia"])
    llm_a = eval_item["variante_a"]["puntuacion_global"]
    llm_b = eval_item["variante_b"]["puntuacion_global"]

    rouge_ganador = "A" if rouge_a > rouge_b else "B"
    llm_ganador = "A" if llm_a > llm_b else "B"
    acuerdo = "✅ acuerdan" if rouge_ganador == llm_ganador else "⚠️  DIFIEREN"
    if rouge_ganador != llm_ganador:
        discrepancias += 1

    print(f"P{eval_item['id']:<3} {rouge_a:>8.3f} {rouge_b:>8.3f} {llm_a:>7.1f} {llm_b:>7.1f}   ROUGE:{rouge_ganador} / LLM:{llm_ganador}  {acuerdo}")

print("-" * 70)
print(f"\n⚠️  Discrepancias ROUGE vs LLM-Judge: {discrepancias}/{len(evaluaciones)} preguntas")
print("\nCuando difieren → confía en el LLM-Judge: mide calidad semántica, no solapamiento léxico")
print("ROUGE alto no significa respuesta buena — solo significa que usa palabras similares a la referencia")

## PASO 8 — Reporte Final y Función de Evaluación Reutilizable

In [ ]:
scores_a = [e["variante_a"]["puntuacion_global"] for e in evaluaciones]
scores_b = [e["variante_b"]["puntuacion_global"] for e in evaluaciones]

print("=" * 55)
print("📊 REPORTE FINAL: PROMPT A (básico) vs PROMPT B (optimizado)")
print("=" * 55)
print(f"\n  Prompt A — media: {np.mean(scores_a):.2f}/10  (min: {min(scores_a)}, max: {max(scores_a)})")
print(f"  Prompt B — media: {np.mean(scores_b):.2f}/10  (min: {min(scores_b)}, max: {max(scores_b)})")
mejora = np.mean(scores_b) - np.mean(scores_a)
print(f"  Mejora del Prompt B:  {mejora:+.2f} puntos")

if mejora > 0:
    print(f"\n  ✅ El prompt optimizado (B) es superior — añadir rol, contexto y")
    print(f"     restricciones de longitud mejora la calidad de las respuestas.")
elif mejora < 0:
    print(f"\n  ❌ Sorpresa: el prompt básico (A) fue mejor en este benchmark.")
    print(f"     Revisa los criterios del juez y el diseño del prompt B.")
else:
    print(f"\n  ➡️  Ambos prompts obtienen la misma puntuación.")

print("\n📋 Detalle por dimensión (Prompt B):")
dims = ["precision_tecnica", "claridad", "completitud"]
for dim in dims:
    scores = [e["variante_b"].get(dim, 0) for e in evaluaciones]
    print(f"  {dim:<22} {np.mean(scores):.1f}/10")

print("\n🔍 Análisis por pregunta:")
for e in evaluaciones:
    diff = e["variante_b"]["puntuacion_global"] - e["variante_a"]["puntuacion_global"]
    emoji = "📈" if diff > 0 else ("📉" if diff < 0 else "➡️")
    print(f"  P{e['id']}: {emoji}  B mejora en {diff:+.0f} pts  |  Fortaleza B: {e['variante_b']['fortalezas'][:60]}")

## La Función Reutilizable: evaluar_pipeline_llm()

### Por qué encapsular la evaluación (PASO 8 de la guía)

Esta función implementa el PASO 8 aplicado a evaluación:  
puedes llamarla con **cualquier chain de LangChain** y cualquier benchmark,  
y te devuelve métricas comparables entre distintas versiones.

Esto es la base del **ciclo de mejora continua** (PASO 7 de la guía):
```
v1 → evaluar_pipeline_llm() → score_v1
   → mejoras el prompt
v2 → evaluar_pipeline_llm() → score_v2
   → compara: ¿v2 > v1?
```

In [ ]:
def evaluar_pipeline_llm(chain, benchmark_items: list, nombre: str = "Pipeline") -> dict:
    """
    Evalúa cualquier chain de LangChain contra un benchmark usando LLM-as-Judge.
    
    Args:
        chain: LangChain chain que acepta {"pregunta": str} y devuelve texto
        benchmark_items: lista de dicts con keys: pregunta, respuesta_referencia, criterios
        nombre: nombre descriptivo del pipeline (para el reporte)
    
    Returns:
        dict con: nombre, score_promedio, score_min, score_max, evaluaciones
    
    Uso:
        resultado_v1 = evaluar_pipeline_llm(chain_a, benchmark, "Prompt Básico v1")
        resultado_v2 = evaluar_pipeline_llm(chain_b, benchmark, "Prompt Optimizado v2")
        print(f"v1: {resultado_v1['score_promedio']} vs v2: {resultado_v2['score_promedio']}")
    """
    resultados = []
    for item in benchmark_items:
        respuesta = chain.invoke({"pregunta": item["pregunta"]})
        evaluacion = evaluar_respuesta(
            item["pregunta"],
            item["respuesta_referencia"],
            respuesta,
            item["criterios"]
        )
        resultados.append(evaluacion)

    scores = [r["puntuacion_global"] for r in resultados]
    return {
        "nombre": nombre,
        "score_promedio": round(np.mean(scores), 2),
        "score_min": min(scores),
        "score_max": max(scores),
        "n_preguntas": len(benchmark_items),
        "evaluaciones": resultados
    }

print("✅ Función evaluar_pipeline_llm() lista para reutilizar en tus proyectos")
print("\nEjemplo de uso:")
print("  resultado = evaluar_pipeline_llm(mi_chain, benchmark, 'Mi Pipeline v3')")
print("  print(f\"Score: {resultado['score_promedio']}/10\")")

## Conclusiones del Notebook T4.6

### Lo que has aprendido

1. **LLM-as-Judge** es el estándar de la industria para evaluar sistemas GenAI:  
   usas un LLM (como juez) para evaluar la calidad de las respuestas de otro LLM.

2. **ROUGE mide similitud léxica, no calidad**: es una métrica clásica de NLP  
   pero insuficiente para GenAI donde las respuestas correctas pueden usar palabras distintas.

3. **A/B test de prompts** (guía PASO 7): siempre compara al menos 2 variantes.  
   Añadir rol + contexto + restricciones de formato casi siempre mejora la calidad.

4. **Temperatura en evaluación = 0**: el juez debe ser determinista.  
   Con temperatura alta, el mismo juez daría puntuaciones diferentes a la misma respuesta.

5. **Encapsulamiento en `evaluar_pipeline_llm()`** (guía PASO 8): convierte la evaluación  
   en una función reutilizable que puedes llamar en cada iteración del ciclo de mejora.

---

### El ciclo de mejora continua completo

```
1. Diseña el prompt inicial (v1)
2. Genera respuestas con chain_v1
3. Evalúa con evaluar_pipeline_llm(chain_v1, benchmark)
4. Analiza: ¿qué preguntas tienen peor puntuación? ¿por qué?
5. Mejora el prompt (v2): añade ejemplos, contexto, restricciones
6. Evalúa con evaluar_pipeline_llm(chain_v2, benchmark)
7. Compara v1 vs v2 → ¿mejoró?
8. Repite desde el paso 4 hasta quedar satisfecho
```

---

### Patrón clave (para memorizar)

```
LLM generador → respuesta → LLM juez → {puntuacion, fortalezas, debilidades}
                                              ↓
                                    Decisión: ¿mejora el prompt o el modelo?
```

---

### Tabla resumen: qué métrica usar en cada caso

| Salida del sistema | Métrica recomendada | Cuándo usarla |
|--------------------|---------------------|---------------|
| Clasificación binaria | Accuracy, F1, AUC | Siempre que tengas labels verdaderos |
| Resúmenes de texto | ROUGE + LLM-Judge | ROUGE como sanity check, LLM para calidad real |
| Respuestas abiertas (QA) | LLM-as-Judge | Lo más frecuente en GenAI |
| Comparación de sistemas | Elo ranking | Muchos candidatos a comparar |
| Sistema en producción | Latencia, coste, error rate | Monitoreo continuo |